# 02_preprocessing

In [1]:
from datasets import load_from_disk
dataset = load_from_disk("../data/processed/iitb_filtered")

In [9]:
[ex["en"] for ex in dataset["train"]]

['Give your application an accessibility workout',
 'Accerciser Accessibility Explorer',
 'The default plugin layout for the bottom panel',
 'The default plugin layout for the top panel',
 'A list of plugins that are disabled by default',
 'Highlight duration',
 'The duration of the highlight box when selecting accessible nodes',
 'Highlight border color',
 'The color and opacity of the highlight border.',
 'Highlight fill color',
 'The color and opacity of the highlight fill.',
 'API Browser',
 'Browse the various methods of the current accessible',
 'Hide private attributes',
 'Method',
 'Property',
 'Value',
 'IPython Console',
 'Interactive console for manipulating currently selected accessible',
 'Event monitor',
 '_ Monitor Events',
 'C _ lear Selection',
 'Everything',
 'Selected application',
 'Selected accessible',
 'Source',
 'Event Monitor',
 'Shows events as they occur from selected types and sources',
 'Highlight last event entry',
 'Start / stop event recording',
 'Clear 

In [8]:
train_en = [ex["en"] for ex in dataset["train"]]
train_hi = [ex["hi"] for ex in dataset["train"]]

In [11]:
import sys
sys.path.append("../src")
from vocab import Vocab

en_vocab = Vocab(min_freq=5)
en_vocab.build(train_en)

hi_vocab = Vocab(min_freq=5)
hi_vocab.build(train_hi)

print(len(en_vocab), len(hi_vocab))

Vocab built: 114395 tokens (min_freq=5, raw unique tokens=473819)
Vocab built: 105200 tokens (min_freq=5, raw unique tokens=555639)
114395 105200


In [13]:
from dataset import TranslationDataset, collate_fn
from torch.utils.data import DataLoader
from functools import partial

train_dataset = TranslationDataset(train_en, train_hi, en_vocab, hi_vocab)
loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                     collate_fn=partial(collate_fn, pad_idx=en_vocab.PAD_IDX))

src, tgt, src_lens, tgt_lens = next(iter(loader))
print(src.shape, tgt.shape)

torch.Size([32, 50]) torch.Size([32, 59])


In [14]:
print(en_vocab.decode(src[0].tolist()))
print(hi_vocab.decode(tgt[0].tolist()))

There should be sufficient space for the <UNK> which should be frequently changed.
हाथी के खड़े होने के लिए पर्याप्त स्थान होना चाहिए और यह निरन्तर बदलते रहना चाहिए।


In [15]:
import pickle
with open("../data/processed/en_vocab.pkl", "wb") as f:
    pickle.dump(en_vocab, f)
with open("../data/processed/hi_vocab.pkl", "wb") as f:
    pickle.dump(hi_vocab, f)